# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yashcodes07/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Final Research Question:**
Given a content item's recent search behavior — 28-day trends in search position, clicks, and click-through rate — can a time-aware classifier predict whether that item is growing, declining, or needs review, in order to rank a content team's manual review queue by priority?

**Decision it supports:** Triage — where a limited content review budget should go first, each week, ranked by model confidence instead of alphabetical order or raw click count.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** FlyRank Internship Pseudonymized Warehouse v20260703 (frozen snapshot, exported 2026-07-03)

**Tables:** fact_content_daily_performance, dim_content, and dim_clients (client selection only)

**Scope:** one pseudonymized client — the one with the longest continuous Search Console history — to keep trend dynamics internally consistent

**Data Window:** two independent 28-day feature windows, each paired with its own following 28-day label window; the two train/test blocks are sequential and non-overlapping in time

**Excluded:**
Rows where gsc_data_available = false
Content items with fewer than 5 observed days in a window
Content items with fewer than 50 total impressions in a window
Unpublished content (is_published = false)

**Privacy:** no client names, domains, URLs, raw exports, or query-level detail appear anywhere; pseudonymous IDs are used only for joining/grouping, never as model features


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

Label definition: normalized 28-day click slope, thresholded into:

1.growing (> +0.02)

2.declining (< −0.02)

3.review (everything else, including high-volatility and missing-trend cases)

A planned fourth "recovering" class produced zero training examples and was folded into "review" — a limitation of the labeling scheme, not the data.

**Features**: avg_position_mean, avg_position_slope, avg_position_volatility, clicks_mean, clicks_slope, ctr_mean, ctr_slope, impressions_total, page_age_days, search_volume, word_count, competition, backlinks
Excluded on purpose: the click-slope value used to derive the label (leakage risk), and client/content IDs (grouping keys only, never features)

**Baseline:** naive majority-class rule — predicts the most common training label ("review") for every test item, blind to test labels (an earlier circular version using test-set label distribution was corrected)
**Model**: Random Forest, 300 trees, max depth 8, class-weighted, random_state=42

**Validation design:** time-aware split — two sequential, non-overlapping 28-day feature+label blocks (train block, then a test block that begins only after the train label window ends). Not a random shuffle/K-fold, since that would leak future-window information.

**Leakage checks:** label-derived slope excluded from features; feature windows never overlap their own or later label windows; IDs never passed as features.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Split	Metric	Score--

Baseline (blind majority class)	F1-macro	0.208

Random Forest	F1-macro	0.380

Random Forest	Accuracy	0.417


---


~83% relative improvement over baseline.


---


Class	Precision	Recall	F1	Support--

declining	0.31	0.15	0.20	747

growing	0.34	0.65	0.45	834

review	0.60	0.42	0.49	1314

Feature importance: avg_position_mean and avg_position_volatility dominate by a wide margin; CTR trend and static content attributes (search volume, word count, competition, backlinks) contribute only secondary signal.

## 5. Limitations

*What this work cannot claim.*

1.Directional signal, not a precise or causal forecast — F1-macro 0.380 means meaningfully better than guessing, not highly accurate

2.Declining pages are hardest to catch: recall of 0.15 means most true declines are missed

3.Growing pages are over-flagged: precision of 0.34 means ~two-thirds of "growing" flags are wrong

4.Single-client scope — results reflect one client's traffic dynamics and aren't claimed to generalize

5.No causal claims — findings are correlational only and say nothing about Google's ranking algorithm

6."Recovering" label was dropped (too few examples to model reliably) — a labeling-scheme limitation, not evidence recovery patterns don't exist

7.No interaction effects tested (e.g., whether static attributes matter more for low-volatility pages) — flagged as a future extension


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

1.Use model output to prioritize, not replace, manual review — start with review-flagged items (strongest precision, 0.60)

2.Pair the model with a simple rule-based safety net for declines, given recall of only 0.15

3.Treat growing flags as a shortlist to verify, not a guarantee (precision 0.34)

4.Prioritize position volatility and mean position as first-class signals in future iterations

5.Re-validate on any new client's own data before extending — results reflect a single client only






## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

1.baseline_vs_model.png — bar chart comparing baseline vs. model F1-macro

2.action_report.csv — the ranked, confidence-sorted action list

3.Per-class precision/recall/F1 table (above)

4.Feature importance ranking (avg_position_mean / avg_position_volatility as top signals)

5.Gap to flag: a metrics JSON (e.g. work/results/test_metrics.json) is not yet committed — currently the results only print to the notebook's stdout, which the reproducibility section notes needs fixing before the "evaluated once, blind" claim is fully checkable from the repo.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
